<a href="https://colab.research.google.com/github/anastasiakalyashova/python-ai-AnastasiaKalyashova/blob/main/week3a_world_map_arcs.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# ═══════════════════════════════════════════════════════
#  ЯЧЕЙКА 0. Подготовка данных (из week2b_read_csv.ipynb)
#  Запускать первой в каждом ноутбуке задания 3
# ═══════════════════════════════════════════════════════

# --- Параметры (изменять здесь) ----------------------
RADIUS_KM     = 300   # радиус соседства гор (для week3a)
TOP_N_ROCKS   = 10    # сколько топ-пород использовать
TOP_N_COMPLEX = 20    # сколько самых «сложных» гор брать
# -----------------------------------------------------

import os, pandas as pd, numpy as np
from itertools import combinations

# 1. Клонируем репозиторий (если ещё нет)
repo = "python-ai-AnastasiaKalyashova"
repo_path = f"/content/{repo}"
if not os.path.exists(repo_path):
    !git clone -q https://github.com/anastasiakalyashova/python-ai-AnastasiaKalyashova.git
if os.getcwd() != repo_path:
    %cd {repo_path}

# 2. Читаем CSV
file_path = None
for root, dirs, files in os.walk("."):
    if "mountains.csv" in files:
        file_path = os.path.join(root, "mountains.csv")
        break
df = pd.read_csv(file_path)

# 3. Переименование столбцов
if "mountainLabel" in df.columns:
    df = df.rename(columns={
        "mountain":          "URL",
        "mountainLabel":     "mountain",
        "rockMaterialLabel": "rockMaterial",
        "elevationMeters":   "elevation",
    })

# 4. Нормализуем породы
df["rockMaterial"] = df["rockMaterial"].str.lower().str.strip()

# 🔧 ИСПРАВЛЕНИЕ: заменяем "lutite" на "пелит"
df["rockMaterial"] = df["rockMaterial"].replace("lutite", "пелит")

# 5. Парсим координаты
coords = df["coordinates"].str.extract(r'Point\(([^\s]+)\s+([^\s]+)\)')
df["lon"] = pd.to_numeric(coords[0], errors="coerce")
df["lat"] = pd.to_numeric(coords[1], errors="coerce")

# 6. df_unique — по одной строке на гору
df_unique = (
    df.groupby("URL")
    .agg(
        mountain   = ("mountain",     "first"),
        lon        = ("lon",          "first"),
        lat        = ("lat",          "first"),
        elevation  = ("elevation",    "first"),
        rock_count = ("rockMaterial", "nunique"),
        rocks      = ("rockMaterial", lambda x: list(x.unique())),
    )
    .reset_index()
)

# 7. df_clean — только физически возможные высоты
df_clean = df_unique[
    (df_unique.elevation >= 0) &
    (df_unique.elevation <= 8849)
].copy()

# 8. Топ пород по частоте (по df_clean)
top_rocks = (
    df[df["URL"].isin(df_clean["URL"])]
    ["rockMaterial"].value_counts()
    .head(TOP_N_ROCKS).index.tolist()
)

# 9. Co-occurrence матрица пород
pairs = []
for rocks in df_clean["rocks"]:
    clean = [r for r in rocks if r in top_rocks]
    pairs += list(combinations(sorted(set(clean)), 2))
cooc = (pd.DataFrame(pairs, columns=["r1", "r2"])
        .value_counts()
        .reset_index(name="count"))

print(f"✅ Длинный формат:    {len(df)} строк")
print(f"✅ Уникальных гор:    {len(df_unique)}")
print(f"✅ df_clean:          {len(df_clean)} гор (0–8849 м)")
print(f"✅ Топ-{TOP_N_ROCKS} пород:    {top_rocks}")
print(f"✅ Пар co-occurrence: {len(cooc)}")

✅ Длинный формат:    4431 строк
✅ Уникальных гор:    2915
✅ df_clean:          2914 гор (0–8849 м)
✅ Топ-10 пород:    ['известняк', 'песчаник', 'гранит', 'мергель', 'конгломерат', 'пелит', 'доломит', 'андезит', 'осадочная горная порода', 'базальт']
✅ Пар co-occurrence: 15


In [ ]:
# ═══════════════════════════════════════════════════════
# ВИЗУАЛИЗАЦИЯ 1: Мировая карта гор с геологическими дугами (УЛУЧШЕННАЯ)
# ═══════════════════════════════════════════════════════

import plotly.express as px
import plotly.graph_objects as go
import numpy as np
import math
from scipy.spatial import cKDTree
from collections import defaultdict

# --- Настройки ----------------------
RADIUS_KM = 300  # можно менять: 100, 300, 500, 1000
# -----------------------------------

print(f"📊 Создаём карту с радиусом соседства {RADIUS_KM} км...")

# 1. Подготовка данных для карты
df_map = df_clean.copy()

# Определяем топ-6 пород + "другое" с подкатегориями
TOP_ROCKS = 6
top_rock_list = top_rocks[:TOP_ROCKS]

# Функция для категоризации с сохранением уникальности "других" пород
def categorize_rock_detailed(rocks_list):
    """Определяем категорию породы с детализацией для редких пород"""
    if not rocks_list:
        return "неизвестно"
    main_rock = rocks_list[0]
    if main_rock in top_rock_list:
        return main_rock
    # Для редких пород возвращаем их настоящее название
    return main_rock

df_map['main_rock'] = df_map['rocks'].apply(categorize_rock_detailed)
df_map['marker_size'] = (df_map['elevation'] / 100 + 5).clip(5, 50)

# Собираем все уникальные породы для цветовой карты
all_rocks = sorted(df_map['main_rock'].unique())
print(f"   Гор на карте: {len(df_map)}")
print(f"   Уникальных пород: {len(all_rocks)}")
print(f"   Топ-{TOP_ROCKS} пород: {top_rock_list}")

# 2. БЫСТРЫЙ поиск пар гор с помощью KD-дерева
print("\n🔍 Быстрый поиск соседних гор (KD-дерево)...")

# Подготовка данных для KD-дерева
earth_radius = 6371

def latlon_to_cartesian(lat, lon):
    """Перевод широты/долготы в 3D координаты"""
    lat_rad = np.radians(lat)
    lon_rad = np.radians(lon)
    x = earth_radius * np.cos(lat_rad) * np.cos(lon_rad)
    y = earth_radius * np.cos(lat_rad) * np.sin(lon_rad)
    z = earth_radius * np.sin(lat_rad)
    return np.column_stack([x, y, z])

cartesian_coords = latlon_to_cartesian(df_map['lat'].values, df_map['lon'].values)
kdtree = cKDTree(cartesian_coords)

# Словарь для подсчёта редкости пород
rock_frequency = df[df["URL"].isin(df_clean["URL"])]["rockMaterial"].value_counts().to_dict()

# Быстрый поиск соседей
mountain_data = []
for idx, row in df_map.iterrows():
    mountain_data.append({
        'idx': idx,
        'url': row['URL'],
        'name': row['mountain'],
        'lat': row['lat'],
        'lon': row['lon'],
        'rocks': set(row['rocks']),
        'elevation': row['elevation']
    })

# Поиск пар с помощью KD-дерева
arcs = []
processed_pairs = set()

for i, mountain in enumerate(mountain_data):
    # Ищем соседей в радиусе RADIUS_KM
    center = cartesian_coords[i]
    indices = kdtree.query_ball_point(center, r=RADIUS_KM)

    for j in indices:
        if j <= i:
            continue

        pair_key = (min(i, j), max(i, j))
        if pair_key in processed_pairs:
            continue
        processed_pairs.add(pair_key)

        m2 = mountain_data[j]

        # Ищем общие породы
        common_rocks = mountain['rocks'] & m2['rocks']

        if common_rocks:
            # Выбираем самую редкую породу
            rarest_rock = min(common_rocks, key=lambda r: rock_frequency.get(r, float('inf')))

            # Вычисляем точное расстояние
            distance = np.linalg.norm(center - cartesian_coords[j])

            arcs.append({
                'lat1': mountain['lat'], 'lon1': mountain['lon'],
                'lat2': m2['lat'], 'lon2': m2['lon'],
                'rock': rarest_rock,
                'distance': distance,
                'name1': mountain['name'], 'name2': m2['name']
            })

    if (i + 1) % 500 == 0:
        print(f"   Обработано {i+1}/{len(mountain_data)} гор, найдено {len(arcs)} дуг")

print(f"✅ Найдено дуг: {len(arcs)}")

# 3. СОЗДАНИЕ ЦВЕТОВОЙ СХЕМЫ (все породы с русскими названиями)
print("\n🎨 Создаём цветовую схему...")

# Расширенная цветовая палитра для всех пород
# Используем качественную палитру Plotly + дополнительные цвета
color_palette = [
    '#FF6B6B', '#4ECDC4', '#45B7D1', '#96CEB4', '#FFEAA7', '#DDA0DD',
    '#98D8C8', '#F7B05E', '#B5EAD7', '#C7CEE6', '#F5A623', '#7B2CBF',
    '#FF9F1C', '#2EC4B6', '#E71D36', '#011627', '#FDFFFC', '#9C89B8',
    '#F0A6CA', '#B8F2E6', '#FFD166', '#06D6A0', '#118AB2', '#073B4C',
    '#EF476F', '#FFD166', '#06D6A0', '#118AB2', '#073B4C', '#F78C6B'
]

# Создаём словарь цветов для каждой породы
color_discrete_map = {}

# Основные породы (топ-6) - яркие, запоминающиеся цвета
main_rock_colors = {
    'известняк': '#FFD700',      # золотой
    'песчаник': '#D2691E',       # шоколадный
    'гранит': '#FF6347',          # томатный
    'мергель': '#9ACD32',         # жёлто-зелёный
    'конгломерат': '#CD853F',     # перу
    'пелит': '#9370DB'            # средне-пурпурный
}

color_discrete_map.update(main_rock_colors)

# Для остальных пород назначаем цвета из палитры
other_rocks = [rock for rock in all_rocks if rock not in main_rock_colors]
for i, rock in enumerate(other_rocks):
    color_discrete_map[rock] = color_palette[i % len(color_palette)]

# Русские названия для легенды (перевод редких пород)
rock_names_ru = {
    'lutite': 'пелит',
    'базальт': 'базальт',
    'андезит': 'андезит',
    'доломит': 'доломит',
    'осадочная горная порода': 'осадочная порода',
    'вулканическая порода': 'вулканическая',
    'метаморфическая горная порода': 'метаморфическая',
    'туф': 'туф',
    'неизвестно': 'неизвестно'
}

# Создаём отображаемые имена для легенды
legend_names = {}
for rock in all_rocks:
    if rock in main_rock_colors:
        legend_names[rock] = rock  # уже на русском
    else:
        legend_names[rock] = rock_names_ru.get(rock, rock)

# 4. СОЗДАНИЕ КАРТЫ
print("🎨 Создаём интерактивную карту...")

# Ограничиваем количество дуг для читаемости
MAX_ARCS = 500
if len(arcs) > MAX_ARCS:
    print(f"   ⚠️ Найдено {len(arcs)} дуг, отображаем только {MAX_ARCS} самых коротких")
    arcs = sorted(arcs, key=lambda x: x['distance'])[:MAX_ARCS]

# Базовая карта с точками
fig = px.scatter_geo(
    df_map,
    lat='lat',
    lon='lon',
    color='main_rock',
    size='marker_size',
    hover_name='mountain',
    hover_data={
        'elevation': ':.0f м',
        'main_rock': 'Порода:',
        'rocks': 'Все породы:',
    },
    title=f'<b>Карта гор с геологическими дугами</b><br>Радиус соседства: {RADIUS_KM} км | Дуг: {len(arcs)}',
    projection='natural earth',
    color_discrete_map=color_discrete_map,
    category_orders={'main_rock': sorted(all_rocks)}
)

# Обновляем названия в легенде на русские
fig.update_traces(
    marker=dict(
        sizemode='area',
        sizeref=2.*max(df_map['marker_size'])/(40**2),
        line=dict(width=0.5, color='white')
    ),
    selector=dict(mode='markers')
)

# Добавляем дуги группами по породам
arcs_by_rock = defaultdict(list)
for arc in arcs:
    arcs_by_rock[arc['rock']].append(arc)

for rock, rock_arcs in arcs_by_rock.items():
    # Создаём линии для каждой породы
    lons = []
    lats = []
    for arc in rock_arcs:
        lons.extend([arc['lon1'], arc['lon2'], None])
        lats.extend([arc['lat1'], arc['lat2'], None])

    # Русское название для легенды дуг
    rock_display = legend_names.get(rock, rock)

    fig.add_trace(go.Scattergeo(
        lon=lons,
        lat=lats,
        mode='lines',
        line=dict(width=1.5, color=color_discrete_map.get(rock, '#A9A9A9')),
        opacity=0.6,
        name=f"{rock_display} (связи)",
        hoverinfo='none',
        showlegend=True
    ))

# Настройка внешнего вида
fig.update_layout(
    height=750,
    margin={"r":0, "t":50, "l":0, "b":0},
    legend_title_text="<b>Тип породы</b>",
    legend=dict(
        yanchor="top",
        y=0.99,
        xanchor="left",
        x=0.01,
        bgcolor="rgba(255, 255, 255, 0.9)",
        bordercolor="Black",
        borderwidth=1,
        font=dict(size=10)
    ),
    title_font_size=16,
    title_x=0.5
)

# Настройка карты
fig.update_geos(
    showcountries=True,
    countrycolor="LightGray",
    showocean=True,
    oceancolor="LightBlue",
    showland=True,
    landcolor="rgb(240, 240, 240)",
    showframe=False,
    showcoastlines=True,
    coastlinecolor="DarkGray"
)

fig.show()

# 5. СТАТИСТИКА
print("\n" + "="*60)
print("📊 СТАТИСТИКА ПО ДУГАМ И ПОРОДАМ")
print("="*60)
print(f"✅ Всего гор на карте: {len(df_map)}")
print(f"✅ Всего дуг: {len(arcs)}")
print(f"✅ Уникальных пород на карте: {len(all_rocks)}")

if len(arcs) > 0:
    print(f"\n🏆 ТОП-10 ПОРОД ПО КОЛИЧЕСТВУ СВЯЗЕЙ:")
    rock_counts = defaultdict(int)
    for arc in arcs:
        rock_counts[arc['rock']] += 1

    for i, (rock, count) in enumerate(sorted(rock_counts.items(), key=lambda x: x[1], reverse=True)[:10], 1):
        rock_name = legend_names.get(rock, rock)
        print(f"   {i}. {rock_name}: {count} дуг ({count/len(arcs)*100:.1f}%)")

    print(f"\n📏 СТАТИСТИКА РАССТОЯНИЙ:")
    distances = [arc['distance'] for arc in arcs]
    print(f"   Среднее расстояние: {np.mean(distances):.0f} км")
    print(f"   Медианное расстояние: {np.median(distances):.0f} км")
    print(f"   Минимальное: {min(distances):.0f} км")
    print(f"   Максимальное: {max(distances):.0f} км")

    print(f"\n🏔️ ТОП-5 САМЫХ СВЯЗАННЫХ ГОР:")
    mountain_connections = defaultdict(int)
    for arc in arcs:
        mountain_connections[arc['name1']] += 1
        mountain_connections[arc['name2']] += 1

    for i, (mountain, count) in enumerate(sorted(mountain_connections.items(), key=lambda x: x[1], reverse=True)[:5], 1):
        print(f"   {i}. {mountain}: {count} связей")
else:
    print("\n⚠️ Дуг не найдено! Попробуйте увеличить RADIUS_KM.")

print("\n💡 Совет: Наведите курсор на точки, чтобы увидеть информацию о горах")
print("   Легенда показывает все породы с их цветами")

📊 Создаём карту с радиусом соседства 300 км...
   Гор на карте: 2914
   Уникальных пород: 87
   Топ-6 пород: ['известняк', 'песчаник', 'гранит', 'мергель', 'конгломерат', 'пелит']

🔍 Быстрый поиск соседних гор (KD-дерево)...
   Обработано 500/2914 гор, найдено 138645 дуг
   Обработано 1000/2914 гор, найдено 253254 дуг
   Обработано 1500/2914 гор, найдено 319024 дуг
   Обработано 2000/2914 гор, найдено 340275 дуг
   Обработано 2500/2914 гор, найдено 347538 дуг
✅ Найдено дуг: 350463

🎨 Создаём цветовую схему...
🎨 Создаём интерактивную карту...
   ⚠️ Найдено 350463 дуг, отображаем только 500 самых коротких



📊 СТАТИСТИКА ПО ДУГАМ И ПОРОДАМ
✅ Всего гор на карте: 2914
✅ Всего дуг: 500
✅ Уникальных пород на карте: 87

🏆 ТОП-10 ПОРОД ПО КОЛИЧЕСТВУ СВЯЗЕЙ:
   1. гранит: 131 дуг (26.2%)
   2. известняк: 81 дуг (16.2%)
   3. конгломерат: 71 дуг (14.2%)
   4. гранодиорит: 44 дуг (8.8%)
   5. пелит: 42 дуг (8.4%)
   6. мергель: 35 дуг (7.0%)
   7. доломит: 22 дуг (4.4%)
   8. алевролит: 13 дуг (2.6%)
   9. аспидный сланец: 13 дуг (2.6%)
   10. мрамор: 7 дуг (1.4%)

📏 СТАТИСТИКА РАССТОЯНИЙ:
   Среднее расстояние: 1 км
   Медианное расстояние: 1 км
   Минимальное: 0 км
   Максимальное: 1 км

🏔️ ТОП-5 САМЫХ СВЯЗАННЫХ ГОР:
   1. Монсеррат: 7 связей
   2. Sant Jeroni: 7 связей
   3. Miranda dels Ecos: 7 связей
   4. Roca Plana dels Llamps: 6 связей
   5. Montgròs: 6 связей

💡 Совет: Наведите курсор на точки, чтобы увидеть информацию о горах
   Легенда показывает все породы с их цветами
